# **Egyptian Legal Project**
- this nootbook is to make a rag pipeline for the project

### **PDF to JSON Pipeline**
- #### **Load PDF** -> **Extract Text** -> **Parse for every rule** -> **save JSON** 

### Import The dependencies

In [55]:
from langchain_community.document_loaders import PyPDFLoader

import re
import json
from pathlib import Path


#### Load PDF

In [56]:
def load_pdf(file_path):
    """Load a PDF file and return its pages as a list of dictionaries."""

    loader  = PyPDFLoader(file_path)
    pages = loader.load()
    return pages

pages = load_pdf("../data/egyptian-civil-code.pdf")
print(f"Number of pages: {len(pages)}")
print("\n===== Page 1 Sample =====\n")
print(pages[0].page_content[:500])

Number of pages: 170

===== Page 1 Sample =====

القانون المدني المصري 
قانون الإصدار 
مادة ١ 
يلغي القانون المدني المعمول به أمام المحاكم الوطنية والصادر في ٨٢ أكتوبر سنة 
٣٨٨١ والقانون المدني المعمول به أمام المحاكم المختلطة والصادر في ٨٢ يونيو 
سنة ٥٧٨١ ويستعاض عنهما بالقانون المدني المرافق لهذا القانون 
مادة ٢ 
على وزير العدل تنفيذ هذا القانون ويعمل به ابتداء من ١٥ أكتوبر سنة 
.٩٤٩١ 
نأمر بأن يبصم هذا القانون بخاتم الدولة وأن ينشر في الجريدة الرسمية وينفذ 
كقانون من قوانين الدولة. 
صدر بقصر القبة في ٩ رمضان سنة ٧٦٣١  ٦١ يوليو سنة ٨٤٩١ 
نصو


#### Extract Articles from pages

In [57]:
full_text = ''
for page in pages:
    full_text += page.page_content + '\n'

In [58]:
def extract_english_articles(full_text):
    """"Extract English articles from the full text of the PDF."""

    # search on english pattern
    en_pattern = r'Article\s+(\d+)\s*\n(.*?)(?=Article\s+\d+|$)'
    en_matches = re.findall(en_pattern, full_text, re.DOTALL)

    english_dict = {}

    for num, text in en_matches:
        article_num = int(num)
        if article_num not in english_dict:
            lines = text.split('\n')
            en_lines = []
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                # break if you get an arabic content
                if re.search(r'[\u0600-\u06FF]', line):
                    break
                en_lines.append(line)
            
            english_text = ' '.join(en_lines).strip()
            if english_text:
                english_dict[article_num] = english_text

    return english_dict

english_dict = extract_english_articles(full_text)
print(f"✅ Total English articles: {len(english_dict)}")
print(f"\nArticle 1: {english_dict.get(1, '')[:150]}")
print(f"\nArticle 2: {english_dict.get(2, '')[:150]}")
print(f"\nArticle 3: {english_dict.get(3, '')[:150]}")

✅ Total English articles: 1070

Article 1: Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, 

Article 2: A provision of a law can only be repealed by a subsequent law expressly providing for such repeal, or containing a provision inconsistent with a provi

Article 3: Periods of limitation will be calculated according to the Gregorian calendar, unless expressly provided otherwise by a law.


In [59]:
def extract_arabic_articles(full_text) -> dict:
    ar_pattern = r'مادة\s*[\(（]?\s*([\d٠-٩]+)[\)）]?\s*[\(（]?\s*\n?(.*?)(?=مادة\s*[\(（]?[\d٠-٩]+|Article\s+\d+|$)'
    ar_matches = re.findall(ar_pattern, full_text, re.DOTALL)
    
    arabic_to_english = str.maketrans('٠١٢٣٤٥٦٧٨٩', '0123456789')
    
    arabic_dict = {}
    for num, text in ar_matches:
        article_num = int(num.translate(arabic_to_english))
        
        if article_num not in arabic_dict:
            lines = text.split('\n')
            ar_lines = []
            for line in lines:
                line = line.strip()
                if not line:
                    continue
                # break if you get an english content
                if not re.search(r'[\u0600-\u06FF]', line):
                    break
                ar_lines.append(line)
            
            arabic_text = ' '.join(ar_lines).strip()
            if arabic_text:
                arabic_dict[article_num] = arabic_text    

    return arabic_dict

# test
arabic_dict = extract_arabic_articles(full_text)
print(f"✅ Total Arabic articles: {len(arabic_dict)}")
print(f"\nArticle 1: {arabic_dict.get(1, '')[:150]}")
print(f"\nArticle 2: {arabic_dict.get(2, '')[:150]}")
print(f"\nArticle 3: {arabic_dict.get(3, '')[:150]}")

✅ Total Arabic articles: 997

Article 1: يلغي القانون المدني المعمول به أمام المحاكم الوطنية والصادر في ٨٢ أكتوبر سنة ٣٨٨١ والقانون المدني المعمول به أمام المحاكم المختلطة والصادر في ٨٢ يونيو

Article 2: على وزير العدل تنفيذ هذا القانون ويعمل به ابتداء من ١٥ أكتوبر سنة .٩٤٩١ نأمر بأن يبصم هذا القانون بخاتم الدولة وأن ينشر في الجريدة الرسمية وينفذ كقانو

Article 3: تحسب المواعيد بالتقويم الميلادي، ما لم ينص هذا القانون على غير ذلك .


In [65]:
def merge_articles(dict) -> list[dict]:
    
    articles = []


    for num in sorted(dict.keys()):
        text = dict.get(num, '')
        
        is_repealed = any(
            w in text 
            for w in ['ملغاة', 'ألغيت', 'Repealed', 'repealed']
        )
        
        articles.append({
            'article_number': num,
            'text':   text,
            'is_repealed':    is_repealed
        })
    
    return articles

english_articles = merge_articles(english_dict)
arabic_articles = merge_articles(arabic_dict)


# Check
for a in english_articles[:5]:
    print(f"\nArticle {a['article_number']}:")
    print(f"  EN_text: {a['text']}")
    print(f"  EN_is_repealed: {a['is_repealed']}")

for a in arabic_articles[:5]:
    print(f"\nArticle {a['article_number']}:")
    print(f"  AR_text: {a['text']}")
    print(f"  AR_is_repealed: {a['is_repealed']}")



Article 1:
  EN_text: Provisions of laws govern all matters to which these provisions apply in letter or spirit. In the absence of a provision of a law that is applicable, the Judge will decide according to custom and in the absence of custom in accordance with the principles of Moslem Law. In the absence of such principles, the Judge will apply the principles of natural justice and the rules of equity.
  EN_is_repealed: False

Article 2:
  EN_text: A provision of a law can only be repealed by a subsequent law expressly providing for such repeal, or containing a provision inconsistent with a provision of the former law or regulating anew a matter previously regulated by a former law.
  EN_is_repealed: True

Article 3:
  EN_text: Periods of limitation will be calculated according to the Gregorian calendar, unless expressly provided otherwise by a law.
  EN_is_repealed: False

Article 4:
  EN_text: A person legitimately exercises his rights is not responsible for prejudice resulting the

## Save JSON

In [66]:
def save_json(articles, output_path):
    """Save the articles to a JSON file."""
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(articles, f, ensure_ascii=False, indent=2)
    print(f"✅ Saved {len(articles)} articles to {output_path}")

In [67]:
output_path_ar = Path("../data/egyptian-civil-code-ar.json")
output_path_en = Path("../data/egyptian-civil-code-en.json")

output_path_ar.parent.mkdir(exist_ok=True)
output_path_en.parent.mkdir(exist_ok=True)

save_json(arabic_articles, output_path_ar)
save_json(english_articles, output_path_en)


✅ Saved 997 articles to ..\data\egyptian-civil-code-ar.json
✅ Saved 1070 articles to ..\data\egyptian-civil-code-en.json
